# PyTorch Geometric Graph Neural Networks with topologic_fast

This notebook demonstrates how to use **topologic_fast** to prepare graph structures for
PyTorch Geometric (PyG) based graph neural networks. We cover end-to-end training,
validation, and testing workflows.

## Overview

1. Create topological structures using topologic_fast
2. Extract graph representations (adjacency, features)
3. Build PyG Data objects
4. Train GNN models for graph classification
5. Visualize results with Plotly

## Prerequisites

```bash
pip install topologic_fast torch torch_geometric pandas plotly scikit-learn pyyaml
```

## Import Libraries

In [ ]:
# Core imports
import topologic_fast as tf
import numpy as np
import pandas as pd
from pathlib import Path
import json
import yaml

# PyTorch and PyG
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, GATv2Conv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.utils import to_undirected

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ML utilities  
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

import torch_geometric
print(f"topologic_fast loaded successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Utility Functions

In [ ]:
def pretty_print_metrics(title: str, metrics: dict) -> None:
    """Pretty print evaluation metrics."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    for k in sorted(metrics.keys()):
        v = metrics[k]
        if isinstance(v, float):
            print(f"{k:30s}: {v:.6f}")
        else:
            print(f"{k:30s}: {v}")
    print("=" * 80 + "\n")

def compute_metrics(y_true, y_pred):
    """Compute classification metrics."""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0)
    }

print("Utilities loaded.")

## Convert topologic_fast Topology to PyG Data

Create functions to convert topological structures to PyTorch Geometric Data objects.

In [ ]:
def topology_to_pyg_data(topology, label=None, node_label_type='categorical', edge_label_type='categorical'):
    """
    Convert a topologic_fast topology to a PyG Data object.
    
    Args:
        topology: CellComplex, Shell, Wire, or other topology
        label: Graph-level label (for classification)
        node_label_type: 'categorical' or 'continuous'
        edge_label_type: 'categorical' or 'continuous'
        
    Returns:
        torch_geometric.data.Data object
    """
    # Create graph from topology
    graph = tf.Graph.ByTopology(topology, direct=True, tolerance=0.001)
    
    vertices = graph.Vertices()
    adj_list = graph.AdjacencyList()
    
    num_nodes = graph.Order()
    
    if num_nodes == 0:
        return None
    
    # Build edge index
    src_nodes = []
    dst_nodes = []
    for i, neighbors in enumerate(adj_list):
        for j in neighbors:
            src_nodes.append(i)
            dst_nodes.append(j)
    
    if len(src_nodes) == 0:
        # Add self-loops for isolated graphs
        edge_index = torch.tensor([[i for i in range(num_nodes)], 
                                    [i for i in range(num_nodes)]], dtype=torch.long)
    else:
        edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)
    
    # Node features: coordinates + topological features
    node_features = []
    for i, v in enumerate(vertices):
        x, y, z = v.Coordinates()
        degree = len(adj_list[i]) if i < len(adj_list) else 0
        # Normalize coordinates
        node_features.append([x, y, z, float(degree)])
    
    x = torch.tensor(node_features, dtype=torch.float32)
    
    # Create Data object
    data = Data(x=x, edge_index=edge_index)
    
    # Add graph label
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.long)
    
    return data


def create_building_cellcomplex(building_type, **kwargs):
    """
    Create a building CellComplex based on type.
    
    Types:
    - 'tower': Tall narrow building
    - 'wide': Wide low building
    - 'linear': Linear arrangement
    - 'courtyard': U-shaped
    - 'grid': Regular grid
    """
    cells = []
    floor_height = kwargs.get('floor_height', 3.0)
    
    if building_type == 'tower':
        w = kwargs.get('width', 2)
        l = kwargs.get('length', 2)
        floors = kwargs.get('floors', 5)
        for f in range(floors):
            z = f * floor_height
            cells.append(tf.Cell.Box(0, 0, z, w, l, floor_height))
            
    elif building_type == 'wide':
        w = kwargs.get('width', 5)
        l = kwargs.get('length', 5)
        floors = kwargs.get('floors', 2)
        for i in range(w):
            for j in range(l):
                for f in range(floors):
                    z = f * floor_height
                    cells.append(tf.Cell.Box(i, j, z, 1, 1, floor_height))
                    
    elif building_type == 'linear':
        length = kwargs.get('length', 6)
        floors = kwargs.get('floors', 2)
        for i in range(length):
            for f in range(floors):
                z = f * floor_height
                cells.append(tf.Cell.Box(i, 0, z, 1, 2, floor_height))
                
    elif building_type == 'courtyard':
        size = kwargs.get('size', 4)
        floors = kwargs.get('floors', 3)
        for f in range(floors):
            z = f * floor_height
            # Left
            cells.append(tf.Cell.Box(0, 0, z, 1, size, floor_height))
            # Right
            cells.append(tf.Cell.Box(size-1, 0, z, 1, size, floor_height))
            # Back
            cells.append(tf.Cell.Box(1, size-1, z, size-2, 1, floor_height))
            
    elif building_type == 'grid':
        nx = kwargs.get('nx', 3)
        ny = kwargs.get('ny', 3)
        nz = kwargs.get('nz', 2)
        for i in range(nx):
            for j in range(ny):
                for k in range(nz):
                    z = k * floor_height
                    cells.append(tf.Cell.Box(i, j, z, 1, 1, floor_height))
    
    if len(cells) > 0:
        return tf.CellComplex.ByCells(cells)
    return None

# Test conversion
print("Testing topology to PyG conversion...")
test_building = create_building_cellcomplex('tower', width=2, length=2, floors=3)
if test_building:
    test_data = topology_to_pyg_data(test_building, label=0)
    print(f"Test data: {test_data}")
    print(f"  Nodes: {test_data.x.shape}")
    print(f"  Edges: {test_data.edge_index.shape}")

## Create Training Dataset

In [ ]:
def generate_pyg_dataset(samples_per_class=30):
    """
    Generate a dataset of building graphs as PyG Data objects.
    
    Classes:
    0: Tower (tall, narrow)
    1: Wide (broad, short)
    2: Linear (elongated)
    3: Courtyard (U-shaped)
    4: Grid (regular)
    """
    data_list = []
    np.random.seed(42)
    
    class_configs = {
        0: ('tower', {'width': (1, 3), 'length': (1, 3), 'floors': (3, 7)}),
        1: ('wide', {'width': (3, 5), 'length': (3, 5), 'floors': (1, 2)}),
        2: ('linear', {'length': (4, 8), 'floors': (1, 3)}),
        3: ('courtyard', {'size': (3, 6), 'floors': (2, 4)}),
        4: ('grid', {'nx': (2, 4), 'ny': (2, 4), 'nz': (1, 3)})
    }
    
    for label, (building_type, param_ranges) in class_configs.items():
        print(f"Generating class {label} ({building_type})...")
        
        for _ in range(samples_per_class):
            # Sample parameters
            params = {}
            for param, (low, high) in param_ranges.items():
                params[param] = np.random.randint(low, high + 1)
            
            try:
                topology = create_building_cellcomplex(building_type, **params)
                if topology is not None:
                    data = topology_to_pyg_data(topology, label=label)
                    if data is not None:
                        data_list.append(data)
            except Exception as e:
                pass  # Skip failed samples
    
    return data_list

# Generate dataset
print("Generating PyG dataset...\n")
dataset = generate_pyg_dataset(samples_per_class=20)
print(f"\nGenerated {len(dataset)} graphs")

# Analyze dataset
labels = [d.y.item() for d in dataset]
print(f"Class distribution: {pd.Series(labels).value_counts().sort_index().to_dict()}")

# Print sample statistics
node_counts = [d.x.shape[0] for d in dataset]
edge_counts = [d.edge_index.shape[1] for d in dataset]
print(f"\nNode count: min={min(node_counts)}, max={max(node_counts)}, mean={np.mean(node_counts):.1f}")
print(f"Edge count: min={min(edge_counts)}, max={max(edge_counts)}, mean={np.mean(edge_counts):.1f}")

## Set Hyperparameters

Configure the model and training parameters.

In [ ]:
# Hyperparameters configuration (similar to topologicpy PyG.SetHyperparameters)
config = {
    # Data splitting
    'cv': 'holdout',
    'split': (0.80, 0.10, 0.10),  # train/val/test
    'random_state': 42,
    'shuffle': True,
    
    # Training
    'epochs': 100,
    'batch_size': 16,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'optimizer': 'adamw',
    'gradient_clip_norm': 1.0,
    'early_stopping': True,
    'early_stopping_patience': 15,
    
    # Model
    'conv': 'sage',  # 'sage', 'gcn', 'gatv2'
    'hidden_dims': (64, 64),
    'activation': 'relu',
    'dropout': 0.2,
    'batch_norm': True,
    'residual': True,
    'pooling': 'mean',  # 'mean', 'max', 'add'
    
    # Task
    'level': 'graph',
    'task': 'classification',
    'num_classes': 5,
    'in_channels': 4  # x, y, z, degree
}

print("Configuration:")
print(json.dumps(config, indent=2))

## Define Graph Neural Network Model

In [ ]:
class GraphClassifier(nn.Module):
    """
    Flexible Graph Neural Network for graph classification.
    Supports multiple convolution types and pooling strategies.
    """
    def __init__(self, config):
        super(GraphClassifier, self).__init__()
        
        self.config = config
        in_channels = config['in_channels']
        hidden_dims = config['hidden_dims']
        num_classes = config['num_classes']
        conv_type = config['conv']
        use_bn = config['batch_norm']
        use_residual = config['residual']
        
        self.use_residual = use_residual
        self.dropout = config['dropout']
        
        # Build convolution layers
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList() if use_bn else None
        self.residual_projs = nn.ModuleList() if use_residual else None
        
        # Select convolution type
        def get_conv(in_c, out_c):
            if conv_type == 'sage':
                return SAGEConv(in_c, out_c)
            elif conv_type == 'gcn':
                return GCNConv(in_c, out_c)
            elif conv_type == 'gatv2':
                return GATv2Conv(in_c, out_c, heads=1)
            else:
                return SAGEConv(in_c, out_c)
        
        # First layer
        self.convs.append(get_conv(in_channels, hidden_dims[0]))
        if use_bn:
            self.bns.append(nn.BatchNorm1d(hidden_dims[0]))
        if use_residual:
            self.residual_projs.append(
                nn.Linear(in_channels, hidden_dims[0]) if in_channels != hidden_dims[0] else nn.Identity()
            )
        
        # Hidden layers
        for i in range(1, len(hidden_dims)):
            self.convs.append(get_conv(hidden_dims[i-1], hidden_dims[i]))
            if use_bn:
                self.bns.append(nn.BatchNorm1d(hidden_dims[i]))
            if use_residual:
                self.residual_projs.append(
                    nn.Linear(hidden_dims[i-1], hidden_dims[i]) if hidden_dims[i-1] != hidden_dims[i] else nn.Identity()
                )
        
        # Pooling
        pooling_type = config['pooling']
        if pooling_type == 'mean':
            self.pool = global_mean_pool
        elif pooling_type == 'max':
            self.pool = global_max_pool
        else:
            self.pool = global_add_pool
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], hidden_dims[-1]),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(hidden_dims[-1], num_classes)
        )
        
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        for i, conv in enumerate(self.convs):
            x_in = x
            x = conv(x, edge_index)
            
            if self.bns is not None:
                x = self.bns[i](x)
            
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            
            if self.use_residual and self.residual_projs is not None:
                x = x + self.residual_projs[i](x_in)
        
        # Global pooling
        x = self.pool(x, batch)
        
        # Classification
        return self.classifier(x)

# Create model
model = GraphClassifier(config).to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")
print(model)

## Split Data and Create DataLoaders

In [ ]:
# Split dataset
train_ratio, val_ratio, test_ratio = config['split']

# Stratified split
labels = [d.y.item() for d in dataset]
indices = list(range(len(dataset)))

train_idx, temp_idx = train_test_split(
    indices, test_size=(val_ratio + test_ratio), 
    random_state=config['random_state'], 
    stratify=labels
)

temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=test_ratio/(val_ratio + test_ratio),
    random_state=config['random_state'],
    stratify=temp_labels
)

train_dataset = [dataset[i] for i in train_idx]
val_dataset = [dataset[i] for i in val_idx]
test_dataset = [dataset[i] for i in test_idx]

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

## Train the Model

In [ ]:
# Setup optimizer
if config['optimizer'] == 'adamw':
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])

criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': []
}

# Early stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print("Training...\n")
for epoch in range(config['epochs']):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        
        # Gradient clipping
        if config['gradient_clip_norm'] > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['gradient_clip_norm'])
        
        optimizer.step()
        
        train_loss += loss.item() * data.num_graphs
        pred = out.argmax(dim=1)
        train_correct += (pred == data.y).sum().item()
        train_total += data.num_graphs
    
    train_loss /= train_total
    train_acc = train_correct / train_total
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data)
            loss = criterion(out, data.y)
            
            val_loss += loss.item() * data.num_graphs
            pred = out.argmax(dim=1)
            val_correct += (pred == data.y).sum().item()
            val_total += data.num_graphs
    
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    # Update history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
    
    if config['early_stopping'] and patience_counter >= config['early_stopping_patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    
print("\nTraining complete!")

## Validate the Model

In [ ]:
model.eval()
val_preds = []
val_labels = []

with torch.no_grad():
    for data in val_loader:
        data = data.to(device)
        out = model(data)
        pred = out.argmax(dim=1)
        val_preds.extend(pred.cpu().tolist())
        val_labels.extend(data.y.cpu().tolist())

val_metrics = compute_metrics(val_labels, val_preds)
pretty_print_metrics("Validation Metrics", val_metrics)

## Test the Model

In [ ]:
model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        pred = out.argmax(dim=1)
        test_preds.extend(pred.cpu().tolist())
        test_labels.extend(data.y.cpu().tolist())

test_metrics = compute_metrics(test_labels, test_preds)
pretty_print_metrics("Test Metrics", test_metrics)

## Plot Training History

In [ ]:
epochs = list(range(1, len(history['train_loss']) + 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Accuracy'))

# Loss curves
fig.add_trace(
    go.Scatter(x=epochs, y=history['train_loss'], mode='lines', name='Train Loss',
               line=dict(color='blue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=epochs, y=history['val_loss'], mode='lines', name='Val Loss',
               line=dict(color='orange')),
    row=1, col=1
)

# Accuracy curves
fig.add_trace(
    go.Scatter(x=epochs, y=history['train_acc'], mode='lines', name='Train Acc',
               line=dict(color='blue')),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=epochs, y=history['val_acc'], mode='lines', name='Val Acc',
               line=dict(color='orange')),
    row=1, col=2
)

fig.update_layout(
    title='Training and Validation Curves',
    height=400, width=900
)
fig.update_xaxes(title_text='Epoch')
fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_yaxes(title_text='Accuracy', row=1, col=2)

fig.show()

## Plot Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
class_names = ['Tower', 'Wide', 'Linear', 'Courtyard', 'Grid']

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=class_names,
    y=class_names,
    colorscale='Blues',
    text=cm,
    texttemplate='%{text}',
    textfont={'size': 14},
    hovertemplate='Actual: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
))

fig.update_layout(
    title='Test Set Confusion Matrix',
    xaxis_title='Predicted Label',
    yaxis_title='Actual Label',
    width=600, height=500
)

fig.show()

## Save the Model

In [ ]:
# Save model
model_path = Path("pyg_model.pt")
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
    'history': history
}, model_path)
print(f"Model saved to {model_path}")

# Phase 2: Prediction on New Data

In [ ]:
def predict_building(topology, model, device):
    """
    Predict the class of a building topology.
    """
    model.eval()
    
    data = topology_to_pyg_data(topology, label=None)
    if data is None:
        return None, None
    
    # Add batch dimension
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long)
    data = data.to(device)
    
    with torch.no_grad():
        logits = model(data)
        probs = F.softmax(logits, dim=1)
        pred = logits.argmax(dim=1).item()
        confidence = probs[0, pred].item()
    
    return pred, confidence

# Test predictions on new buildings
print("Predicting on new buildings...\n")

test_buildings = [
    ('tower', {'width': 2, 'length': 2, 'floors': 6}),
    ('wide', {'width': 4, 'length': 4, 'floors': 1}),
    ('linear', {'length': 7, 'floors': 2}),
    ('courtyard', {'size': 5, 'floors': 3}),
    ('grid', {'nx': 3, 'ny': 3, 'nz': 2})
]

class_names = ['Tower', 'Wide', 'Linear', 'Courtyard', 'Grid']

for building_type, params in test_buildings:
    topology = create_building_cellcomplex(building_type, **params)
    if topology:
        pred, conf = predict_building(topology, model, device)
        print(f"{building_type.capitalize():12s} -> Predicted: {class_names[pred]:12s} (confidence: {conf:.2f})")

## Notes on topologic_fast Integration

### Key API Differences from topologicpy

| topologicpy | topologic_fast |
|-------------|----------------|
| `from topologicpy.PyG import PyG` | `import topologic_fast as tf` |
| `PyG.ByCSVPath(path, ...)` | Use `topology_to_pyg_data()` helper |
| `pyg.SetHyperparameters(...)` | Define config dict manually |
| `pyg.Train()` | Standard PyTorch training loop |
| `pyg.Validate()` | Standard PyTorch evaluation |
| `pyg.PlotHistory()` | Use Plotly directly |
| `pyg.PlotConfusionMatrix()` | Use Plotly directly |

### Features Not Yet Implemented

```python
# NOT YET AVAILABLE in topologic_fast - requires manual implementation:
# PyG.ByCSVPath() - Load datasets from CSV
# PyG.SetHyperparameters() - Use config dict
# PyG.Train() - Use standard PyTorch loop
# PyG.Validate() - Use standard evaluation
# PyG.Test() - Use standard evaluation
# PyG.Predict() - Use model.forward()
# PyG.PlotHistory() - Use Plotly
# PyG.PlotConfusionMatrix() - Use Plotly
# PyG.SaveModel() - Use torch.save()
# PyG.LoadModel() - Use torch.load()
```

### Benefits of topologic_fast

1. **Fast graph extraction**: 10-100x faster for large topologies
2. **Efficient memory usage**: Rust-based memory management
3. **Consistent API**: Method-based syntax for all topology types
4. **Thread-safe**: Parallel dataset generation supported

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")